<a href="https://colab.research.google.com/github/ekuelkpodar/Complex-Systems-Google-Colab-Experiment/blob/main/Global_Scientific_Collaboration_Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Global Scientific Collaboration Network
This notebook builds a network of researchers and papers using open scientific APIs.

In [1]:
!pip install pyvis networkx pandas requests plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 34.7 MB/s eta 0:00:00


In [3]:
import networkx as nx
import pandas as pd
import requests
import time
from pyvis.network import Network
import plotly.express as px

def fetch_semantic_scholar_data(query, limit=20, retries=3):
    """Fetches paper data from Semantic Scholar API with retry logic."""
    base_url = 'https://api.semanticscholar.org/graph/v1/paper/search'
    params = {
        'query': query,
        'limit': limit,
        'fields': 'title,authors,year,citationCount'
    }

    for i in range(retries):
        response = requests.get(base_url, params=params)
        if response.status_code == 200:
            return response.json().get('data', [])
        elif response.status_code == 429:
            print(f"Rate limited. Retrying in {2**i} seconds...")
            time.sleep(2**i)
        else:
            print(f"Error: {response.status_code}")
            break

    # Fallback sample data if API fails
    print("API unavailable. Using sample data for demonstration.")
    return [
        {'title': 'AI in Science', 'authors': [{'name': 'A. Smith'}, {'name': 'B. Jones'}], 'year': 2023, 'citationCount': 50},
        {'title': 'Neural Networks', 'authors': [{'name': 'B. Jones'}, {'name': 'C. Wang'}], 'year': 2022, 'citationCount': 120},
        {'title': 'Deep Learning', 'authors': [{'name': 'A. Smith'}, {'name': 'C. Wang'}], 'year': 2024, 'citationCount': 30}
    ]

# Initial search
papers = fetch_semantic_scholar_data('artificial intelligence', limit=20)
print(f'Processed {len(papers)} papers.')

Rate limited. Retrying in 1 seconds...
Rate limited. Retrying in 2 seconds...
Rate limited. Retrying in 4 seconds...
API unavailable. Using sample data for demonstration.
Processed 3 papers.


In [4]:
def build_network(papers):
    G = nx.Graph()

    for paper in papers:
        authors = [auth.get('name') for auth in paper.get('authors', []) if auth.get('name')]
        # Create nodes for authors
        for author in authors:
            if not G.has_node(author):
                G.add_node(author, title=author, size=10)

        # Create edges between co-authors
        from itertools import combinations
        for auth1, auth2 in combinations(authors, 2):
            if G.has_edge(auth1, auth2):
                G[auth1][auth2]['weight'] += 1
            else:
                G.add_edge(auth1, auth2, weight=1)

    return G

# Build and visualize the graph
G = build_network(papers)

net = Network(notebook=True, height='500px', width='100%', bgcolor='#222222', font_color='white', cdn_resources='remote')
net.from_nx(G)

# Calculate some metrics
centrality = nx.degree_centrality(G)
top_researchers = sorted(centrality.items(), key=lambda x: x[1], reverse=True)[:5]

print(f'Network created with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.')
print('Top Influential Researchers (Degree Centrality):', top_researchers)

# Save and display visualization
net.show('collaboration_network.html')

Network created with 3 nodes and 3 edges.
Top Influential Researchers (Degree Centrality): [('A. Smith', 1.0), ('B. Jones', 1.0), ('C. Wang', 1.0)]
collaboration_network.html


## Community Detection and Impact Analysis
In this section, we identify research clusters and visualize the citation impact of the papers.

In [5]:
from networkx.algorithms import community

# Detect communities using the Louvain method (greedy modularity communities)
communities = community.greedy_modularity_communities(G)
print(f"Found {len(communities)} research clusters (communities).")

for i, comm in enumerate(communities):
    print(f"Cluster {i+1}: {', '.join(comm)}")

# Prepare data for Plotly visualization of paper influence
paper_df = pd.DataFrame(papers)
fig = px.scatter(paper_df, x='year', y='citationCount', text='title',
                 size='citationCount', color='citationCount',
                 title='Scientific Influence: Citations vs Year',
                 labels={'citationCount': 'Citations', 'year': 'Publication Year'})
fig.update_traces(textposition='top center')
fig.show()

Found 1 research clusters (communities).
Cluster 1: C. Wang, B. Jones, A. Smith
